# FAIR Organelle Segmentation pipeline

In [ ]:
import sys
sys.path.append('..')

# Relies on https://github.com/volume-em/empanada-napari.git@inf_pipeline_dev
from empanada_napari._slice_inference import SliceInferenceWidget
import glob
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
from napari.components import ViewerModel
import numpy as np
import ome_zarr
from random import random
from skimage import measure
from skimage.transform import resize

from src.fair_segmentation.image_util import *

## Load source data

In [ ]:
source_path = 'D:/slides/AMC_EM/**/*.tif'

filenames = glob.glob(source_path)
filenames


## Process data

In [ ]:
downscale = 1

datas = []
for filename in filenames:
    data, metadata, pixel_size = extract_tiff_olympus(filename)
    if downscale != 1:
        new_shape = list(data.shape)
        new_shape[0] //= downscale
        new_shape[1] //= downscale
        data = resize(data, new_shape)
    datas.append(float2int_image(norm_image_quantiles(data)))

## Select data

In [ ]:
%matplotlib inline

data = datas[0]

plt.imshow(data, cmap='gray')

## Run model

In [ ]:
viewer = ViewerModel()
image_layer = viewer.add_image(data)

nclass_objects = 10000
inference_config = SliceInferenceWidget(viewer=viewer,
                                        image_layer=image_layer,
                                        maximum_objects_per_class=nclass_objects,
                                        model_config='MitoNet_v1',
                                        downsampling=2,
                                       )

seg, axis, plane, y, x = inference_config.config_and_run_inference(use_thread=False)

## Extract instances

In [ ]:
# Get all instances of class #1 (single class)
classes = seg // nclass_objects
instances = seg % nclass_objects

print("#classes:", np.max(classes))
print("#instances:", np.max(instances))

## Show instance stats

In [ ]:
regions = measure.regionprops(instances)
print("id\tarea")
for props in regions:
    if props.area > 10000:
        print(f"{props.label}\t{int(props.area)}")

## Draw output

In [ ]:
maxval = np.max(instances)
colors = [(0,0,0)] + [(random(),random(),random()) for _ in range(maxval)]
new_map = LinearSegmentedColormap.from_list('random', colors, N=maxval+1)
plt.imshow(data, cmap='gray')
plt.imshow(instances, cmap=new_map, alpha=0.75)